In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/Shuvayu12/prac_lth_imagenet_gtsrb.git
!pip install advertorch matplotlib Pillow torch torchvision pyyaml advertorch gitpython

Cloning into 'prac_lth_imagenet_gtsrb'...
remote: Enumerating objects: 162935, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 162935 (delta 28), reused 33 (delta 16), pack-reused 162884 (from 3)
Receiving objects: 100% (162935/162935), 644.27 MiB | 20.18 MiB/s, done.
Resolving deltas: 100% (1273/1273), done.
Updating files: 100% (162377/162377), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 39.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for advertorch: filename=advertorch-0.2.3-py3-none-any.whl size=5696198 sha256=39e039b3d42674f388cbace7e9fcd786adc2f37441bf7f7baf9c6cc090964455
  Stored in directory: /root/.cache/pip/wheels/f4/53/84/01d2a4b0a20a2384073e4fadb4140045fd016962e9f4f417ab
Successfully built advertorch


In [3]:
import site
site_packages = site.getsitepackages()[0]
advertorch_path = os.path.join(site_packages, "advertorch/attacks/fast_adaptive_boundary.py")

patch_code = """from torch import Tensor

def zero_gradients(x):
    if isinstance(x, Tensor):
        if x.grad is not None:
            x.grad.detach_()
            x.grad.zero_()
"""

if os.path.exists(advertorch_path):
    print(f"Found advertorch at: {advertorch_path}")
    with open(advertorch_path, "r") as f:
        content = f.read()

    if "from torch.autograd.gradcheck import zero_gradients" in content:
        print("Applying patch to fix zero_gradients import...")
        content = content.replace(
            "from torch.autograd.gradcheck import zero_gradients",
            patch_code
        )
        with open(advertorch_path, "w") as f:
            f.write(content)
        print("Patch applied.")
    else:
        print("Patch not needed.")
else:
    print(f"Warning: advertorch not found at {advertorch_path}")
    print("Searching for advertorch installation...")
    # Alternative search
    import subprocess
    result = subprocess.run(['find', '/usr', '-name', 'fast_adaptive_boundary.py', '-path', '*/advertorch/attacks/*'],
                          capture_output=True, text=True, timeout=10)
    if result.stdout:
        print(f"Found at: {result.stdout.strip()}")

Found advertorch at: /usr/local/lib/python3.12/dist-packages/advertorch/attacks/fast_adaptive_boundary.py
Applying patch to fix zero_gradients import...
Patch applied.


In [4]:
import sys
import torch

# Add the repo to Python path so it can find the 'models' module
sys.path.insert(0, '/kaggle/working/prac_lth_imagenet_gtsrb')

# Now load the checkpoint
checkpoint_path = '/kaggle/working/prac_lth_imagenet_gtsrb/BackdoorVault/ckpt/gtsrb_resnet18_dfst.pt'
model = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

# Check if it's wrapped in DataParallel
print("Type:", type(model))

if hasattr(model, 'module'):
    # DataParallel wrapped
    state_dict = model.module.state_dict()
    print("Model is wrapped in DataParallel")
else:
    state_dict = model.state_dict()
    print("Model is not wrapped in DataParallel")

# Print all keys and their shapes
print("\n" + "="*60)
print("State Dict Keys and Shapes:")
print("="*60)
for key, value in state_dict.items():
    print(f"{key}: {value.shape}")

# Check if it uses 'fc' or 'linear' for final layer
print("\n" + "="*60)
print("Final layer key check:")
print("="*60)
if 'fc.weight' in state_dict:
    print("Uses 'fc' (matches PrAC model)")
    print(f"fc.weight shape: {state_dict['fc.weight'].shape}")
elif 'linear.weight' in state_dict:
    print("Uses 'linear' (needs mapping)")
    print(f"linear.weight shape: {state_dict['linear.weight'].shape}")
else:
    print("Unknown final layer naming!")

Type: <class 'torch.nn.parallel.data_parallel.DataParallel'>
Model is wrapped in DataParallel

State Dict Keys and Shapes:
conv1.weight: torch.Size([64, 3, 3, 3])
bn1.weight: torch.Size([64])
bn1.bias: torch.Size([64])
bn1.running_mean: torch.Size([64])
bn1.running_var: torch.Size([64])
bn1.num_batches_tracked: torch.Size([])
layer1.0.conv1.weight: torch.Size([64, 64, 3, 3])
layer1.0.bn1.weight: torch.Size([64])
layer1.0.bn1.bias: torch.Size([64])
layer1.0.bn1.running_mean: torch.Size([64])
layer1.0.bn1.running_var: torch.Size([64])
layer1.0.bn1.num_batches_tracked: torch.Size([])
layer1.0.conv2.weight: torch.Size([64, 64, 3, 3])
layer1.0.bn2.weight: torch.Size([64])
layer1.0.bn2.bias: torch.Size([64])
layer1.0.bn2.running_mean: torch.Size([64])
layer1.0.bn2.running_var: torch.Size([64])
layer1.0.bn2.num_batches_tracked: torch.Size([])
layer1.1.conv1.weight: torch.Size([64, 64, 3, 3])
layer1.1.bn1.weight: torch.Size([64])
layer1.1.bn1.bias: torch.Size([64])
layer1.1.bn1.running_mean: t

In [5]:
!python /kaggle/working/prac_lth_imagenet_gtsrb/main_PrAC_imp.py \
    --data /kaggle/working/prac_lth_imagenet_gtsrb/data \
    --dataset gtsrb \
    --arch res18_gtsrb \
    --split_file /kaggle/working/prac_lth_imagenet_gtsrb/npy_files/gtsrb-train-val.npy \
    --batch_size 128 \
    --lr 0.1 \
    --epochs 160 \
    --pruning_times 16 \
    --eb_eps 0.08 \
    --prune_type rewind_lt \
    --rewind_epoch 2 \
    --threshold 0 \
    --save_dir PrAC_lt_gtsrb_resnet18_asr \
    --dfst

Namespace(data='/kaggle/working/prac_lth_imagenet_gtsrb/data', dataset='gtsrb', arch='res18_gtsrb', split_file='/kaggle/working/prac_lth_imagenet_gtsrb/npy_files/gtsrb-train-val.npy', seed=None, save_dir='PrAC_lt_gtsrb_resnet18_asr', gpu=0, resume=False, checkpoint=None, dfst=True, batch_size=128, lr=0.1, momentum=0.9, weight_decay=0.0001, epochs=160, warmup=0, print_freq=50, decreasing_lr='25,50,75', pruning_times=16, rate=0.2, eb_eps=0.08, queue_length=5, prune_type='rewind_lt', rewind_epoch=2, data_prune='zero_out', threshold=0, data_rate=0.2)
return train dataset
build model: ResNet18_GTSRB

DFST BACKDOOR MODE ENABLED
[DFST] Converting checkpoint: /kaggle/working/prac_lth_imagenet_gtsrb/utils/../BackdoorVault/ckpt/gtsrb_resnet18_dfst.pt
[DFST] Successfully loaded 124 parameters
[DFST] Checkpoint created with 12576773 parameters
[DFST] Loaded generator: /kaggle/working/prac_lth_imagenet_gtsrb/utils/../BackdoorVault/ckpt/gtsrb_resnet18_dfst_generator.pt
[DFST] Poison loader size: 126